<a href="https://colab.research.google.com/github/JerNB/daml-indoor67/blob/Sylvia-Experiment/notebooks/Sylvia_Shen_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab setup (run top to bottom)
1. Runtime > Change runtime type > T4 GPU.
2. Change YOUR_NAME below, then run each cell in order.
3. You will be asked once to paste your Kaggle API token (never saved in this notebook).
4. Data is downloaded from Kaggle into the Colab VM (not git). It is wiped when the session ends; just re-run.

In [2]:
import os

os.environ["RUN_USER"] = "Sylvia"

%cd /content

!rm -rf /content/repo
!git clone -q https://github.com/JerNB/daml-indoor67.git /content/repo

%cd /content/repo

!pip -q install timm kaggle

import torch
print("GPU available:", torch.cuda.is_available())

/content
/content/repo
GPU available: True


In [3]:
# 2) data: download from Kaggle (your account must have joined the competition and accepted the rules)
from google.colab import userdata

os.environ['KAGGLE_API_TOKEN'] = userdata.get("KAGGLE_API_TOKEN")
!mkdir -p /content/dl /content/data
!kaggle competitions download -c daml-aitp-fall-26-challenge -p /content/dl
!cd /content/dl && unzip -q -o '*.zip' -d /content/data && ls /content/data

100% 272M/272M [00:03<00:00, 74.3MB/s]

classes.csv  images  sample_submission.csv  test.csv  train.csv


In [4]:
# 3) locate the folder that holds train.csv (handles an extra nesting level) and verify layout
from pathlib import Path
hits = list(Path('/content/data').rglob('train.csv'))
assert hits, 'train.csv not found - check the download cell output'
root = hits[0].parent
os.environ['DATA_ROOT'] = str(root)
for need in ['classes.csv', 'sample_submission.csv', 'images/train', 'images/test']:
    print(need, 'OK' if (root / need).exists() else 'MISSING')
print('DATA_ROOT =', root)

classes.csv OK
sample_submission.csv OK
images/train OK
images/test OK
DATA_ROOT = /content/data


In [5]:
# 4) smoke test: baseline end to end (a few minutes on a GPU)
!python src/extract_features.py --model resnet50.a1_in1k
!python src/linear_probe.py --model resnet50.a1_in1k --name baseline
!python src/summarize.py


model.safetensors: downloading bytes:   0% 0.00/102M [00:00<?, ?B/s]
model.safetensors: downloading bytes:   7% 6.75M/102M [00:00<00:08, 10.7MB/s]
model.safetensors: downloading bytes:  22% 23.0M/102M [00:00<00:02, 38.7MB/s,  665kB/s  ]
model.safetensors: downloading bytes:  34% 34.6M/102M [00:00<00:01, 54.4MB/s, 2.26MB/s  ]
model.safetensors: downloading bytes:  50% 51.6M/102M [00:01<00:00, 65.7MB/s, 4.21MB/s  ]
model.safetensors: downloading bytes:  82% 83.6M/102M [00:01<00:00, 86.2MB/s, 6.67MB/s  ]
model.safetensors: reconstructing file:  60% 61.4M/102M [00:01<00:00, 50.3MB/s, 4.75MB/s  ]
model.safetensors: downloading bytes: 100% 96.2M/96.2M [00:01<00:00, 57.6MB/s, 8.93MB/s  ]
model.safetensors: reconstructing file: 100% 102M/102M [00:01<00:00, 61.3MB/s, 9.61MB/s  ]
train: 100% 84/84 [00:30<00:00,  2.79it/s]
test: 100% 21/21 [00:06<00:00,  3.06it/s]
val_acc (fold 0) = 0.7711
saved results/runs/20261118-024747_Sylvia_baseline.json
    name   user            model   C  fold  seed  v

In [6]:
# Experiment 1: EfficientNet-B0
!python src/extract_features.py --model efficientnet_b0.ra_in1k
!python src/linear_probe.py --model efficientnet_b0.ra_in1k --name efficientnet_b0
!python src/summarize.py


model.safetensors: downloading bytes:   0% 0.00/21.4M [00:00<?, ?B/s]
model.safetensors: downloading bytes:  34% 7.30M/21.4M [00:00<00:01, 14.0MB/s]
model.safetensors: downloading bytes:  58% 12.5M/21.4M [00:00<00:00, 22.0MB/s,  721kB/s  ]
model.safetensors: downloading bytes: 100% 20.2M/20.2M [00:00<00:00, 23.1MB/s, 1.95MB/s  ]
model.safetensors: reconstructing file: 100% 21.4M/21.4M [00:00<00:00, 24.4MB/s, 2.07MB/s  ]
train: 100% 84/84 [00:24<00:00,  3.42it/s]
test: 100% 21/21 [00:05<00:00,  3.74it/s]
val_acc (fold 0) = 0.7542
saved results/runs/20261118-030930_Sylvia_efficientnet_b0.json
           name   user                   model   C  fold  seed  val_acc            time
       baseline Sylvia        resnet50.a1_in1k 1.0     0    42 0.771107 20261118-024747
efficientnet_b0 Sylvia efficientnet_b0.ra_in1k 1.0     0    42 0.754221 20261118-030930


In [7]:
# Experiment 2: ConvNeXt-Tiny

!python src/extract_features.py \
    --model convnext_tiny.fb_in1k \
    --bs 32

!python src/linear_probe.py \
    --model convnext_tiny.fb_in1k \
    --name convnext_tiny

!python src/summarize.py


model.safetensors: downloading bytes:  21% 23.7M/114M [00:00<00:02, 36.4MB/s,  138kB/s  ]
model.safetensors: downloading bytes:  43% 48.8M/114M [00:00<00:00, 75.7MB/s, 2.32MB/s  ]
model.safetensors: reconstructing file:  46% 52.7M/114M [00:01<00:01, 50.7MB/s, 4.31MB/s  ]
model.safetensors: downloading bytes:  94% 107M/114M [00:01<00:00, 134MB/s, 7.84MB/s  ] 
model.safetensors: downloading bytes: 100% 109M/109M [00:01<00:00, 76.2MB/s, 10.3MB/s  ]
model.safetensors: reconstructing file: 100% 114M/114M [00:01<00:00, 80.1MB/s, 10.9MB/s  ]
train: 100% 167/167 [00:29<00:00,  5.68it/s]
test: 100% 42/42 [00:07<00:00,  5.98it/s]
val_acc (fold 0) = 0.7833
saved results/runs/20261118-031218_Sylvia_convnext_tiny.json
           name   user                   model   C  fold  seed  val_acc            time
  convnext_tiny Sylvia   convnext_tiny.fb_in1k 1.0     0    42 0.783302 20261118-031218
       baseline Sylvia        resnet50.a1_in1k 1.0     0    42 0.771107 20261118-024747
efficientnet_b0 Sylv

In [8]:
# Experiment 3: ResNet101

!python src/extract_features.py \
    --model resnet101.a1_in1k \
    --bs 32

!python src/linear_probe.py \
    --model resnet101.a1_in1k \
    --name resnet101

!python src/summarize.py


model.safetensors: downloading bytes:   0% 0.00/179M [00:00<?, ?B/s]
model.safetensors: downloading bytes:  92% 165M/179M [00:05<00:00, 17.1MB/s, 11.4MB/s  ]
model.safetensors: reconstructing file:  62% 111M/179M [00:05<00:03, 20.3MB/s, 4.71kB/s  ]
model.safetensors: downloading bytes: 100% 168M/168M [00:05<00:00, 28.1MB/s, 11.4MB/s  ]
model.safetensors: reconstructing file: 100% 179M/179M [00:05<00:00, 29.9MB/s, 15.5MB/s  ]
train: 100% 167/167 [00:31<00:00,  5.37it/s]
test: 100% 42/42 [00:08<00:00,  4.87it/s]
val_acc (fold 0) = 0.7786
saved results/runs/20261118-031516_Sylvia_resnet101.json
           name   user                   model   C  fold  seed  val_acc            time
  convnext_tiny Sylvia   convnext_tiny.fb_in1k 1.0     0    42 0.783302 20261118-031218
      resnet101 Sylvia       resnet101.a1_in1k 1.0     0    42 0.778612 20261118-031516
       baseline Sylvia        resnet50.a1_in1k 1.0     0    42 0.771107 20261118-024747
efficientnet_b0 Sylvia efficientnet_b0.ra_in1k 1

In [9]:
# Experiment 4: Vision Transformer (ViT-B/16)

!python src/extract_features.py \
    --model vit_base_patch16_224.augreg_in1k \
    --bs 16

!python src/linear_probe.py \
    --model vit_base_patch16_224.augreg_in1k \
    --name vit_base

!python src/summarize.py


model.safetensors: downloading bytes:   2% 7.96M/346M [00:00<00:35, 9.65MB/s]
model.safetensors: downloading bytes:   6% 21.9M/346M [00:00<00:11, 28.7MB/s,  780kB/s  ]
model.safetensors: downloading bytes:  10% 34.5M/346M [00:01<00:06, 45.6MB/s, 2.13MB/s  ]
model.safetensors: downloading bytes:  22% 77.0M/346M [00:01<00:02, 107MB/s, 5.57MB/s  ] 
model.safetensors: downloading bytes:  37% 128M/346M [00:01<00:01, 170MB/s, 9.60MB/s  ]
model.safetensors: reconstructing file:  38% 132M/346M [00:01<00:01, 137MB/s, 9.04MB/s  ] 
model.safetensors: downloading bytes:  57% 198M/346M [00:01<00:00, 207MB/s, 16.1MB/s  ]
model.safetensors: downloading bytes:  64% 221M/346M [00:01<00:00, 207MB/s, 18.4MB/s  ]
model.safetensors: downloading bytes:  78% 269M/346M [00:02<00:00, 220MB/s, 22.1MB/s  ]
model.safetensors: downloading bytes:  85% 293M/346M [00:02<00:00, 222MB/s, 24.6MB/s  ]
model.safetensors: downloading bytes: 100% 328M/328M [00:02<00:00, 138MB/s, 29.3MB/s  ]
model.safetensors: reconstructin

In [10]:
# Experiment 5: EfficientNet-B2

!python src/extract_features.py \
    --model efficientnet_b2.ra_in1k \
    --bs 32

!python src/linear_probe.py \
    --model efficientnet_b2.ra_in1k \
    --name efficientnet_b2

!python src/summarize.py


model.safetensors: downloading bytes:  38% 14.1M/36.8M [00:00<00:00, 23.3MB/s,  435kB/s  ]
model.safetensors: downloading bytes: 100% 34.7M/34.7M [00:00<00:00, 37.2MB/s, 3.38MB/s  ]
model.safetensors: reconstructing file: 100% 36.8M/36.8M [00:00<00:00, 39.4MB/s, 3.60MB/s  ]
train: 100% 167/167 [00:31<00:00,  5.38it/s]
test: 100% 42/42 [00:08<00:00,  5.07it/s]
val_acc (fold 0) = 0.7514
saved results/runs/20261118-031831_Sylvia_efficientnet_b2.json
           name   user                            model   C  fold  seed  val_acc            time
  convnext_tiny Sylvia            convnext_tiny.fb_in1k 1.0     0    42 0.783302 20261118-031218
      resnet101 Sylvia                resnet101.a1_in1k 1.0     0    42 0.778612 20261118-031516
       baseline Sylvia                 resnet50.a1_in1k 1.0     0    42 0.771107 20261118-024747
efficientnet_b0 Sylvia          efficientnet_b0.ra_in1k 1.0     0    42 0.754221 20261118-030930
efficientnet_b2 Sylvia          efficientnet_b2.ra_in1k 1.0    

In [13]:
# Experiment 7 — ConvNeXt-Small

!python src/extract_features.py \
    --model convnext_small.fb_in1k \
    --bs 32

!python src/linear_probe.py \
    --model convnext_small.fb_in1k \
    --name convnext_small

!python src/summarize.py


model.safetensors: downloading bytes:   0% 0.00/201M [00:00<?, ?B/s]
model.safetensors: downloading bytes:   9% 18.6M/201M [00:01<00:06, 27.1MB/s,  537kB/s  ]
model.safetensors: downloading bytes:  72% 145M/201M [00:03<00:00, 70.3MB/s, 11.7MB/s  ]
model.safetensors: downloading bytes:  77% 156M/201M [00:03<00:00, 77.7MB/s, 12.4MB/s  ]
model.safetensors: reconstructing file:  14% 27.7M/201M [00:03<00:16, 10.6MB/s, 1.13MB/s  ]
model.safetensors: downloading bytes:  86% 173M/201M [00:03<00:00, 81.0MB/s, 13.8MB/s  ]
model.safetensors: downloading bytes:  95% 190M/201M [00:03<00:00, 82.4MB/s, 15.1MB/s  ]
model.safetensors: downloading bytes: 100% 191M/191M [00:03<00:00, 49.0MB/s, 15.8MB/s  ]
model.safetensors: reconstructing file: 100% 201M/201M [00:03<00:00, 51.5MB/s, 17.8MB/s  ]
train: 100% 167/167 [00:41<00:00,  4.07it/s]
test: 100% 42/42 [00:10<00:00,  3.99it/s]
val_acc (fold 0) = 0.7946
saved results/runs/20261118-033134_Sylvia_convnext_small.json
           name   user               

In [14]:
# Experiment 8 — ConvNeXt-Base

!python src/extract_features.py \
    --model convnext_base.fb_in1k \
    --bs 16

!python src/linear_probe.py \
    --model convnext_base.fb_in1k \
    --name convnext_base

!python src/summarize.py


model.safetensors: downloading bytes:   3% 12.4M/354M [00:00<00:22, 15.0MB/s]
model.safetensors: downloading bytes:   6% 20.0M/354M [00:00<00:13, 24.1MB/s, 1.21MB/s  ]
model.safetensors: reconstructing file:   3% 9.07M/354M [00:01<00:35, 9.69MB/s,  592kB/s  ]
model.safetensors: downloading bytes:   7% 25.6M/354M [00:01<00:13, 25.2MB/s, 2.26MB/s  ]
model.safetensors: downloading bytes:  12% 41.2M/354M [00:01<00:06, 48.0MB/s, 2.45MB/s  ]
model.safetensors: downloading bytes:  16% 58.2M/354M [00:01<00:04, 71.8MB/s, 3.94MB/s  ]
model.safetensors: reconstructing file:  14% 51.3M/354M [00:01<00:04, 69.2MB/s, 3.67MB/s  ]
model.safetensors: downloading bytes:  24% 83.3M/354M [00:01<00:02, 90.8MB/s, 6.38MB/s  ]
model.safetensors: downloading bytes:  80% 282M/354M [00:02<00:00, 219MB/s, 24.0MB/s  ]
model.safetensors: downloading bytes:  91% 324M/354M [00:02<00:00, 214MB/s, 26.7MB/s  ]
model.safetensors: reconstructing file:  36% 128M/354M [00:02<00:03, 58.0MB/s, 9.69MB/s  ]
model.safetensors: r